# Safari Compass Calibration

The Safari-Zone analog of the **Metronome Compass Calibration** notebook.  It
identifies the loaded seed, plans the manual advances needed to encounter a
Metang, identifies the *battle* seed from the safari encounter (bait / mud /
ball), checks how confident that identification is, saves the run, and feeds the
shared timer→frame calibration model.

## Two seeds, two kinds of "frame" (read this first)

Both seeds are fixed by **game-frame (clock) timing** — the timer precision we
calibrate:

- **Seed A** — the overworld stream: encounters, roamer relocation, Elm calls.
- **Seed B** — the battle stream: hits, crits, capture / flee odds.

An **advance frame** ("advance") is how many times a seed's state has been
advanced via `advance_rng`, driven by **player actions, not the clock**.  Section A
walks *Seed A's* advance frame (Elm calls + chatot flips + Sweet Scent) purely so
that we encounter a Metang — this does **not** affect Seed B or the calibration.
Calibration is the same timer(M)→Seed-B-frame fit as metronome; safari just
identifies Seed B differently and may carry a slightly different load-screen
offset, applied as a separate **safari offset** (β/slope stays from metronome).

## Sections
- **A** — identify Seed A (roamer + Elm), then plan the advances to a Metang.
- **B** — identify Seed B via safari compass, then a confidence / neighbor check.
- **C** — save the run to `data/safari_runs.jsonl`.
- **D** — analysis over the saved runs.
- **E** — apply the safari offset to `data/calibration_model.json` (offset only).

In [5]:
%load_ext autoreload
%autoreload 2
import datetime as dt

from utils.calibration_tools import (
    # Section A -- roamer routes + Elm seed identification (shared with metronome)
    generate_roamer_candidates_near,
    print_roamer_candidates,
    identify_seed,
    # Section C -- persist a safari run
    save_safari_run,
    # Section D -- analysis
    load_safari_runs,
    fit_safari_offset,
    # Section E -- apply the safari offset (deliberate; review-then-confirm)
    update_safari_offset,
)
from utils.safari_advance import (
    advance_context, context_from_row,
    identify_frame, prompt_target_frame,
    plan_advances, margin_guide, describe_plan,
)
from utils.safari_confidence import path_confidence, print_confidence

from claytonlib.compass import compass_safari, CompassSafariInput
from claytonlib.calibration import CalibrationModel
from claytonlib.safari import safari_pokemon_by_name
from claytonlib.chart import STRATEGY_ONLY_BALLS, CRITERIA_CAPTURE

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Section A.1 — Roamer + Elm identification  (→ `a_seed`)

Same as the metronome notebook's Section A.  Configure the target datetime/delay,
the search window, and each roamer's **current** route (before the reset).  After
loading the save, read the roamer map + Elm phone to pin the seed.

In [12]:
# --- Section A.1: roamer / Elm target + current roamer state ---
a_target_time    = dt.datetime(2025, 7, 24, 14, 45, 55)   # <-- your load datetime
a_target_delay   = 681                                     # <-- your load delay
a_seconds_window = 1        # +/- X seconds
a_delay_window   = 60       # +/- Y delays
a_match_parity   = True     # only delays with target_delay's even/odd parity
a_display_limit  = 40       # rows to print (None = all)

# Each roamer's CURRENT route (before the reset).  A roamer roams iff it appears here.
a_prev_routes = {"r": 46, "e": 34, "l": 24}

a_candidates = generate_roamer_candidates_near(
    a_target_time, a_target_delay, a_seconds_window, a_delay_window,
    prev_routes=a_prev_routes, match_parity=a_match_parity,
)

# Interactively pin down the seed: roamer routes -> Elm calls -> (M) manual pick.
a_seed = identify_seed(a_candidates, display_limit=a_display_limit)
a_seed

Observed roamer routes (R E L, space-separated, . = any):  29 38 19



Observed R=29 E=38 L=19  ->  3 / 183 candidate(s) match

3 candidate seed(s)

        Seed                 Time   Delay    dD   ds    R   E   L   #  Elm
  0x0B0E02C8  2025-07-24 14:45:54     687    +6   -1   29  38  19   3  PEEPKKPKKPEEPPP
  0x0C0E02C8  2025-07-24 14:45:55     687    +6   +0   29  38  19   3  KKPKKPKPPEKPKEE
  0x0D0E02C8  2025-07-24 14:45:56     687    +6   +1   29  38  19   3  PPKEKEPEEKPPEPE


Elm calls (type P/E/K as heard; M = pick manually):  kpk


Elm calls so far: KPK
2 candidate seed(s)

        Seed                 Time   Delay    dD   ds    R   E   L   #  Elm
  0x0B0E02C8  2025-07-24 14:45:54     687    +6   -1   29  38  19   3  PEEPKKPKKPEEPPP
  0x0C0E02C8  2025-07-24 14:45:55     687    +6   +0   29  38  19   3  KKPKKPKPPEKPKEE


Elm calls (type P/E/K as heard; M = pick manually):  k


...

Elm calls (type P/E/K as heard; M = pick manually):  p


Elm calls so far: KPKKP
2 candidate seed(s)

        Seed                 Time   Delay    dD   ds    R   E   L   #  Elm
  0x0B0E02C8  2025-07-24 14:45:54     687    +6   -1   29  38  19   3  PEEPKKPKKPEEPPP
  0x0C0E02C8  2025-07-24 14:45:55     687    +6   +0   29  38  19   3  KKPKKPKPPEKPKEE


Elm calls (type P/E/K as heard; M = pick manually):  k


Elm calls so far: KPKKPK
1 candidate seed(s)

        Seed                 Time   Delay    dD   ds    R   E   L   #  Elm
  0x0C0E02C8  2025-07-24 14:45:55     687    +6   +0   29  38  19   3  KKPKKPKPPEKPKEE

=== Seed identified: 0x0C0E02C8  2025-07-24 14:45:55  delay=687  R/E/L=29/38/19  Elm=KKPKKPKPPEKPKEE ===


{'seed': 202244808,
 'time': datetime.datetime(2025, 7, 24, 14, 45, 55),
 'delay': 687,
 'sec_delta': 0,
 'delay_delta': 6,
 'r_route': 29,
 'e_route': 38,
 'l_route': 19,
 'rng_calls': 3,
 'elm': 'KKPKKPKPPEKPKEE',
 'elm_list': ['K',
  'K',
  'P',
  'K',
  'K',
  'P',
  'K',
  'P',
  'P',
  'E',
  'K',
  'P',
  'K',
  'E',
  'E']}

## Section A.2 — Advance planning  (→ how to reach a Metang)

We are **not** guaranteed a Metang, so we walk Seed A's *advance frame* to one
that yields a Metang (frame 81 = the shiny Metang when we hit the target seed
exactly; otherwise use Pokefinder to pick a Metang frame).

1. **Identify the current advance frame** from the Elm calls you've heard so far
   (1 Elm call = 1 advance).  `max_offset` assumes you paused within ~15 advances
   of the roamer relocation.
2. **Pick the target frame** (Pokefinder handoff — paste the printed Seed A into
   Pokefinder, find a Metang frame, type it back; blank = 81).
3. **Plan the advances**: bulk via chatot flips (2 advances each), then a
   verifiable margin of Elm calls, then Sweet Scent.  The guide shows the Elm
   calls to expect around the target — `]!` marks where to Sweet Scent.

In [13]:
# --- Section A.2: locate the current advance frame, then plan to the target ---
# Regenerate a long Elm sequence for THIS seed (covers the approach to frame ~81+).
a_rng_calls, a_elm = advance_context(a_seed["seed"], a_prev_routes, count=160)

# Type the Elm calls you've heard since the reset (P/E/K); prompts until unique.
a_current_frame = identify_frame(a_rng_calls, a_elm, observed="", max_offset=15)

# Pokefinder handoff for the target encounter frame (blank keeps 81 = shiny Metang).
a_target_frame = prompt_target_frame(a_seed["seed"], default=81)

a_plan  = plan_advances(a_current_frame, a_target_frame)   # margin defaults to 3 Elm calls
a_guide = margin_guide(a_rng_calls, a_elm, a_plan)
print(describe_plan(a_plan, a_guide))

Elm calls: (none)  ->  16 possible frames: [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]


More Elm calls (type P/E/K as heard):  KPKKPK


Elm calls: KPKKPK  ->  advance frame 10
Seed A: 0x0C0E02C8  -- find a Metang encounter frame in Pokefinder.


Target encounter frame [81]:  31


On advance frame 10; want a Metang encounter on frame 31 (Sweet Scent while on frame 31).
  Advances to go: 21
  1. 9 chatot flips (18 advances) -> land on frame 28
  2. 3 Elm calls -> frame 31, then Sweet Scent.
  Guide: PKPKE[KPE]!EKK   (]! = Sweet Scent here)


## Section B — Safari-compass Seed-B identification  (→ `b_matched`)

Drives the **calibrated** `compass_safari` (frame center from the model, ±kσ over
the RTC-second offsets), exactly as `expedition.compass_safari` does.  Walk the
safari encounter turn by turn — enter `m`/`b`/ball-shakes/`F`/`C` as you see them
— until the candidate set narrows.  Then a confidence check scans for other
nearby seeds that reproduce the same path (aliases), ranked by distance.

The boot key seed and initial time come straight from Section A's identified
`a_seed` (the loaded seed and its datetime) -- no need to re-enter them.  `b_M`
is the commanded countdown = `target_timer_delay + target_timer_calibration`.

In [ ]:
# --- Section B: calibrated safari-compass target ---
b_key_seed              = a_seed["seed"]                   # the loaded Seed A (from Section A)
b_initial_time          = a_seed["time"]                   # its datetime (from identify_seed)
b_target_timer_delay    = 327792                           # <-- commanded timer delay (ms)
b_target_timer_calibration = 0                             # <-- timer calibration (ms, signed)
b_max_target_seconds    = 600                              # <-- chart's max target (s)
b_pokemon_name          = "metang"
b_second_offsets        = (-1, 0, 1)   # cover off-by-one timer-start timing (the "3 seconds")
b_confidence_frame_range = 100          # +/- frames to scan for path-aliases

b_M = b_target_timer_delay + b_target_timer_calibration
model = CalibrationModel.load_default()   # deployed linear model (data/calibration_model.json)

b_inputs = CompassSafariInput.from_expedition_target(
    model=model, M=b_M, initial_time=b_initial_time, key_seed=b_key_seed,
    max_target_seconds=b_max_target_seconds,
    pokemon=safari_pokemon_by_name(b_pokemon_name),
    strategy=STRATEGY_ONLY_BALLS, criteria=CRITERIA_CAPTURE,
    second_offsets=b_second_offsets, mass_cap=0.999,
)

# Interactive: enter the safari path as you play it out.
b_matched = compass_safari(b_inputs)
b_matched

=== Compass: Safari Zone Seed Identifier ===
  m      Mud, no crit                  Metang is angry!
  M / a  Mud, crit (Anger)             Metang is beside itself with anger!
  b      Bait, no crit                 Metang is eating!
  B / e  Bait, crit (Eating)           Metang is busy eating!
  0      Ball, 0 shakes                Oh, no! The Pokémon broke free!
  1      Ball, 1 shake                 Aww! It appeared to be caught!
  2      Ball, 2 shakes                Aargh! Almost had it!
  3      Ball, 3 shakes                Shoot! It was so close, too!
  C      Captured (ends)               Gotcha! Metang was caught!
  F      Fled (ends)                   Metang fled!
  u      Undo last action              —
  ?x     Uncertain result              —
  J      Switch to Jane                —
  w      Widen window & re-apply path  —
  Spaces and commas in input are ignored.


Seeds: 1283 / 1283 remaining
Path:  (none)
Balls: 30
   #        Seed    Frame       Δ   δsec   P(land)
   1.


>>  b



Seeds: 1160 / 1283 remaining
Path:  b
Balls: 30
   #        Seed    Frame       Δ   δsec   P(land)
   1. 0xF70E4E61    20065      +1      0     0.43%
   2. 0xF70E4E5F    20063      -1      0     0.43%
   3. 0xF70E4E62    20066      +2      0     0.43%
   4. 0xF70E4E63    20067      +3      0     0.43%
   5. 0xF70E4E5D    20061      -3      0     0.43%
  Most likely: 0xF70E4E61  P=0.43%  (timer on time)



>>  b



Seeds: 788 / 1283 remaining
Path:  bb
Balls: 30
   #        Seed    Frame       Δ   δsec   P(land)
   1. 0xF70E4E62    20066      +2      0     0.64%
   2. 0xF70E4E63    20067      +3      0     0.64%
   3. 0xF70E4E5D    20061      -3      0     0.64%
   4. 0xF70E4E5C    20060      -4      0     0.64%
   5. 0xF70E4E65    20069      +5      0     0.64%
  Most likely: 0xF70E4E62  P=0.64%  (timer on time)


...

  Now ±15.1606σ over offsets [-2, -1, 0, 1, 2]: 1859 candidates; re-applied 10 observed step(s).

Seeds: 0 / 1859 remaining
Path:  bbbbBb0001
Balls: 30
   #        Seed    Frame       Δ   δsec   P(land)

The observed path eliminated every candidate in the current window (±15.1606σ, σ≈68.6, offsets [-2, -1, 0, 1, 2]).  Expand the search to look further out, or fix an input error.
  Current window: ±1040 frames (k=15.1606σ, σ≈68.6), ±2s.


  Expand frames by how many (each side)? [0]  1000
  Expand seconds by how many (each side)? [0]  1


  Now ±29.7364σ over offsets [-3, -2, -1, 0, 1, 2, 3]: 1860 candidates; re-applied 10 observed step(s).

Seeds: 0 / 1860 remaining
Path:  bbbbBb0001
Balls: 30
   #        Seed    Frame       Δ   δsec   P(land)

The observed path eliminated every candidate in the current window (±29.7364σ, σ≈68.6, offsets [-3, -2, -1, 0, 1, 2, 3]).  Expand the search to look further out, or fix an input error.
  Current window: ±2040 frames (k=29.7364σ, σ≈68.6), ±3s.


  Expand frames by how many (each side)? [0]  2000
  Expand seconds by how many (each side)? [0]  0


  Now ±58.888σ over offsets [-3, -2, -1, 0, 1, 2, 3]: 1860 candidates; re-applied 10 observed step(s).

Seeds: 0 / 1860 remaining
Path:  bbbbBb0001
Balls: 30
   #        Seed    Frame       Δ   δsec   P(land)

The observed path eliminated every candidate in the current window (±58.888σ, σ≈68.6, offsets [-3, -2, -1, 0, 1, 2, 3]).  Expand the search to look further out, or fix an input error.
  Current window: ±4040 frames (k=58.888σ, σ≈68.6), ±3s.


In [ ]:
# --- Section B: confidence / neighbor check ---
# Re-enter the observed path (compass_safari doesn't return it); reused when saving below.
b_observed_path = input("Observed safari path (m/b/0-3/F/C): ").strip()

if len(b_matched) == 1:
    b_seed = int(b_matched[0], 16)
    b_neighbors = path_confidence(b_inputs, b_seed, b_observed_path,
                                  frame_range=b_confidence_frame_range)
    print_confidence(b_neighbors)
else:
    b_seed = None
    print(f"{len(b_matched)} seeds still matched -- narrow further before trusting a single seed.")

## Section C — Save the run  (→ `data/safari_runs.jsonl`)

Appends this run — the identified Seed B (only when a single seed matched), the
Section-A `a_seed` (so the offset fit has `F_a`), the observed path, the commanded
timer, and the calibrated landing (frame / RTC second / δ) — via the existing
`save_safari_run`.  Prompts for tag / timer fields / notes and confirms before
writing.  Saving does **not** touch the calibration model (that's Section E).

In [ ]:
# --- Section C: append this run to data/safari_runs.jsonl ---
run_record = save_safari_run(b_matched, inputs=b_inputs, a_seed=a_seed, path=b_observed_path)

## Section D — Analysis over `safari_runs.jsonl`

Sparse for now.  Shows the run count and previews the safari **offset** the
current runs imply against the deployed model (does *not* write it).  The
safari-vs-metronome offset measurement proper is tracked in `clayton-abf.10`.

In [ ]:
# --- Section D: quick look at the collected safari runs ---
runs = load_safari_runs()
confident = [r for r in runs if r.get("seed") is not None]
print(f"{len(runs)} safari run(s) saved; {len(confident)} with a confident single seed.")

fit = fit_safari_offset(model)   # holds the model slope; median residual = the offset
if fit:
    print(f"Safari offset preview: {fit['offset']:+.2f} frames "
          f"(n={fit['n']}, std={fit['std']:.2f})  -- not written until Section E.")
else:
    print("No usable runs yet (need a_seed + a confident single seed).")

## Section E — Apply the safari offset  (→ `data/calibration_model.json`)

Re-fits the safari **offset only** (holding the metronome slope/β) from
`safari_runs.jsonl`, shows the old → new offset per model, and writes it **only
after you confirm**.  It sets a *separate* `safari_offset` field — the metronome
`alpha`/`beta` are untouched — so the metronome/chart path is unchanged and **no
chart rebuild is needed**; a chart report opts in via `use_safari_offset`.

In [ ]:
# --- Section E: review the safari offset re-fit, then write it only if confirmed ---
new_models = update_safari_offset()